# semantic_search/02 — Cluster patients in each embedding space

KMeans-clusters each feature file from `01_aggregate`, saves the labels, and measures how much the
different note types agree about how the cohort divides.

**Runs after** `semantic_search/01_aggregate`. **Runs before** `semantic_search/03_characterize`.

## Preprocessing, in this order

```
L2-normalize rows  ->  StandardScaler  ->  PCA(50)  ->  KMeans(k)
```

**L2 first** because transformer embeddings are trained in a cosine geometry; on unit-norm rows
Euclidean KMeans is monotone in cosine distance, so KMeans optimizes the similarity the model
actually encodes. **StandardScaler** then stops a few high-variance dimensions from dominating —
this matters most for `concat`, where three blocks of different scale sit side by side. **PCA**
denoises and makes the silhouette scan affordable.

## Choosing k

`K_CHOSEN = None` takes the silhouette argmax over `K_RANGE`. Silhouette on high-dimensional
embeddings is a weak criterion and usually prefers small k — the scan CSV and the plot below are
there so you can override per space rather than trusting the argmax. Set `K_CHOSEN = {"merged": 6}`
to pin one, or `--k` to pin all.

**Labels are relabeled by ascending cluster size**, so cluster 0 is always the smallest. KMeans
label integers are otherwise an artifact of centroid initialization order, and a rerun could produce
the same partition under permuted names — which would silently make every stage-3 per-cluster table
incomparable across runs.

## The concordance table

`cluster_concordance.csv` is the adjusted Rand index between every pair of runs, on their shared
patients. This is the arm's central question in one number: **a low ARI between the pathology and
imaging partitions means the note types carve the cohort differently**, rather than all three
restating one dominant axis (which, on real data, is most likely cancer type).

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "semantic_search").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402
from semantic_search import common  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns the missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<24} {path}")
    print(f"\n{'All inputs present.' if not missing else str(len(missing)) + ' missing: ' + ', '.join(missing)}")
    return missing


def run_module(module: str, args: list[str] | None = None) -> int:
    """Run a semantic_search stage as a subprocess, streaming its output."""
    cmd = [sys.executable, "-m", module] + (args or [])
    print("$ " + " ".join(cmd) + "\n", flush=True)
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f"\nexit={proc.returncode}  elapsed={time.time() - t0:,.1f}s", flush=True)
    return proc.returncode


print(f"repo root:  {REPO_ROOT}")
print(f"data root:  {config.DATA_PATH}")
print(f"this arm:   {config.SEMANTIC_SEARCH_PATH}")

## Configuration

`RANDOM_SEED`, `N_COMPONENTS` and `SILHOUETTE_MAX_N` mirror module constants in
`cluster_patients.py` — change them there, not here.

Coordinates for stage 3's scatter plots come from the **first two principal components**, not UMAP:
`umap-learn` is not in `environment.yml`, and PCA keeps the scatter in the same geometry the
clustering actually used.

In [ ]:
MODULE = "semantic_search.cluster_patients"

SPACES = common.SPACES
WINDOWS = common.WINDOWS
K_MIN, K_MAX = 2, 12
K_CHOSEN = None    # None -> silhouette argmax per space; int -> force that k everywhere
SEED = 0
OVERWRITE = False

print(f"spaces:  {SPACES}")
print(f"windows: {WINDOWS}")
print(f"k scan:  {K_MIN}..{K_MAX}   chosen: {K_CHOSEN or 'silhouette argmax'}")

## Preconditions

Every feature file this run intends to cluster.

In [ ]:
missing = check_inputs([
    (f"{s}/{w}", common.feature_path(s, w)) for w in WINDOWS for s in SPACES
])
if missing:
    print("\nRun semantic_search/01_aggregate.ipynb first for the missing spaces.")

## Run

In [ ]:
args = ["--spaces", *SPACES, "--windows", *WINDOWS,
        "--k-min", str(K_MIN), "--k-max", str(K_MAX), "--seed", str(SEED)]
if K_CHOSEN is not None:
    args += ["--k", str(K_CHOSEN)]
if OVERWRITE:
    args.append("--overwrite")

rc = run_module(MODULE, args)
if rc != 0:
    print("\nStage failed - see the traceback above.")

## Silhouette vs k

Look for an elbow or a local peak rather than the global max. A curve that falls monotonically from
k=2 means the space has no strong cluster structure at any k — a real and reportable result for an
exploratory arm, not a failure.

In [ ]:
import polars as pl

scan_path = common.result_path("silhouette_scan")
if os.path.exists(scan_path):
    scan = pl.read_csv(scan_path)
    for window in WINDOWS:
        sub = scan.filter(pl.col("window") == window)
        if not sub.height:
            continue
        print(f"\n[{window}]  silhouette by k")
        for space in SPACES:
            row = sub.filter(pl.col("space") == space).sort("k")
            if not row.height:
                continue
            cells = "  ".join(f"k{r['k']}={r['silhouette']:.3f}"
                              for r in row.iter_rows(named=True))
            best = row.sort("silhouette", descending=True).row(0, named=True)
            print(f"  {space:10s} {cells}   -> best k={best['k']}")
else:
    print(f"No scan at {scan_path} - has the run completed?")

## Chosen partitions

In [ ]:
import json

for window in WINDOWS:
    print(f"\n[{window}]")
    for space in SPACES:
        path = common.cluster_meta_path(space, window)
        if not os.path.exists(path):
            print(f"  {space:10s} (no labels)")
            continue
        meta = common.load_cluster_meta(space, window)
        sizes = [meta["cluster_sizes"][k] for k in sorted(meta["cluster_sizes"], key=int)]
        print(f"  {space:10s} k={meta['k']:<3d} n={meta['n_patients']:>6,d}  "
              f"sil={meta['silhouette']:.3f}  pca_var={meta['pca_variance']:.1%}  sizes={sizes}")

## Do the note types agree?

ARI = 1 means two partitions are identical; ARI = 0 means they agree no better than chance.

- **Same space, `alltime` vs `pretreatment`** — high ARI means the anchor barely matters.
- **Different spaces, same window** — this is the real question. High ARI everywhere means one
  dominant axis (check stage 3: usually cancer type) is driving all of them. Low ARI means the note
  types genuinely see different patients, which is the more interesting outcome.

In [ ]:
conc_path = common.result_path("cluster_concordance")
if os.path.exists(conc_path):
    conc = pl.read_csv(conc_path)

    same_space = conc.filter(pl.col("space_a") == pl.col("space_b"))
    if same_space.height:
        print("Same space, alltime vs pretreatment:")
        for r in same_space.sort("ari", descending=True).iter_rows(named=True):
            print(f"  {r['space_a']:10s} ARI={r['ari']:.3f}  n_shared={r['n_shared']:,}")

    for window in WINDOWS:
        cross = conc.filter(
            (pl.col("window_a") == window) & (pl.col("window_b") == window)
            & (pl.col("space_a") != pl.col("space_b"))
        )
        if not cross.height:
            continue
        print(f"\nDifferent spaces, both {window}:")
        for r in cross.sort("ari", descending=True).iter_rows(named=True):
            print(f"  {r['space_a']:10s} vs {r['space_b']:10s} "
                  f"ARI={r['ari']:.3f}  n_shared={r['n_shared']:,}")
else:
    print(f"No concordance table at {conc_path}")

## Next

`semantic_search/03_characterize.ipynb` asks what these clusters correspond to clinically.